# Apartado 3 — Validación y Optimización de Hiperparámetros (Versión 60/20/20)

En este apartado dividimos el conjunto de datos `music_classification.csv` en:

- **60% entrenamiento**
- **20% validación**
- **20% test**

El objetivo es estudiar el comportamiento del modelo `DecisionTreeClassifier`
al variar los hiperparámetros:

- `min_samples_leaf`
- `ccp_alpha`

A diferencia del apartado 2, aquí NO usamos todo el dataset para entrenar.
Cada combinación de hiperparámetros:

1. se entrena **solo con training**,  
2. se evalúa **solo con validation**,  
3. se selecciona la que obtenga **mejor F1 en el conjunto de validación**.

Finalmente:

- se reentrena el modelo óptimo con **train + validation**,  
- y se evalúa su rendimiento final en **test**.

Esto sigue exactamente la metodología de selección de hiperparámetros descrita en el Tema 7.


In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("music_classification.csv")
X = df.drop(columns=["Class"]).values
y = df["Class"].values

# 60% train + 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=0, stratify=y
)

# dividir 40% en 20/20 → 50/50
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=0, stratify=y_temp
)

len(X_train), len(X_val), len(X_test)


(10797, 3599, 3600)

## Exploración manual de hiperparámetros

El enunciado exige probar distintas combinaciones de los hiperparámetros:

- `min_samples_leaf ∈ {1, 5, 10, 20}`
- `ccp_alpha ∈ {0.0, 0.001, 0.01, 0.05}`

Entrenaremos un árbol para cada combinación usando **SOLO entrenamiento**
y evaluaremos su F1 en **validación**.


In [11]:
from sklearn.metrics import f1_score
from sklearn.tree import DecisionTreeClassifier, plot_tree


param_grid = {
    "min_samples_leaf": [1, 5, 10, 20],
    "ccp_alpha": [0.0, 0.001, 0.01, 0.05]
}

results = []

for m in param_grid["min_samples_leaf"]:
    for a in param_grid["ccp_alpha"]:
        tree = DecisionTreeClassifier(
            min_samples_leaf=m,
            ccp_alpha=a,
            random_state=0
        ).fit(X_train, y_train)

        preds_val = tree.predict(X_val)
        f1 = f1_score(y_val, preds_val, average="macro")

        results.append({
            "min_samples_leaf": m,
            "ccp_alpha": a,
            "f1_val": f1,
            "n_nodes": tree.tree_.node_count,
            "n_leaves": tree.get_n_leaves()
        })

df_results = pd.DataFrame(results)
df_results.sort_values("f1_val", ascending=False)





,min_samples_leaf,ccp_alpha,f1_val,n_nodes,n_leaves
12,20,0.000,0.446850,805,403
4,5,0.000,0.430104,3077,1539
8,10,0.000,0.428990,1615,808
0,1,0.000,0.397781,7831,3916
1,1,0.001,0.396129,73,37
5,5,0.001,0.396129,73,37
9,10,0.001,0.396129,73,37
13,20,0.001,0.396129,73,37
2,1,0.010,0.212816,9,5
6,5,0.010,0.212816,9,5


In [14]:
best_row = df_results.sort_values("f1_val", ascending=False).iloc[0]

# --- convertir a tipos válidos para sklearn ---
best_m = int(best_row["min_samples_leaf"])
best_a = float(best_row["ccp_alpha"])

print("Mejor combinación encontrada:")
best_row


Mejor combinación encontrada:


min_samples_leaf     20.00000
ccp_alpha             0.00000
f1_val                0.44685
n_nodes             805.00000
n_leaves            403.00000
Name: 12, dtype: float64

## Interpretación de los resultados de validación (60% Train – 20% Val – 20% Test)

La siguiente tabla muestra el rendimiento del modelo para todas las combinaciones
de hiperparámetros probadas. Cada modelo se entrenó únicamente con el conjunto
de entrenamiento (60%) y se evaluó sobre el conjunto de validación (20%),
tal como exige el enunciado.



A partir de esta tabla podemos extraer las siguientes conclusiones:

---

### 1. El mejor modelo según validación:  
## **min_samples_leaf = 20**  
## **ccp_alpha = 0.0**

Este modelo obtiene **F1_val = 0.44685**, el mayor de todas las combinaciones probadas.

---

### 2. Tendencias observadas con respecto a `min_samples_leaf`

Cuando mantenemos `ccp_alpha = 0`:

- `min_samples_leaf = 1` → Árbol gigante (7831 nodos) → sobreajuste → F1_val bajo (0.397).
- Aumentar este valor simplifica el árbol, reduce sobreajuste y mejora F1_val.
- **El máximo F1_val se alcanza con 20**, el valor más alto probado.

Esto confirma la conclusión del Apartado 2:  
> aumentar `min_samples_leaf` reduce la complejidad del árbol y mejora la generalización.

---

### 3. Tendencias observadas con respecto to `ccp_alpha`

Independientemente de `min_samples_leaf`:

- Con `alpha = 0.001`: todos los modelos caen a **73 nodos**  
  → poda muy agresiva  
  → todos dan exactamente **0.39612** de F1_val.
- Con `alpha = 0.01`: el árbol cae a solo **9 nodos** → infraajuste severo.
- Con `alpha = 0.05`: árbol trivial de **1 nodo** → F1_val = 0.039.

Esto coincide perfectamente con el Experimento 2:
> la poda por complejidad de CART es muy agresiva incluso para valores pequeños de α.

Por tanto, **los mejores valores de F1_val se obtienen sin poda global**:  
`ccp_alpha = 0`.

---

### 4. Árboles demasiado pequeños → F1_val extremadamente bajo

Los modelos con α ≥ 0.01 tienen:

- menos de 10 nodos,
- entre 1 y 5 hojas,
- y un F1_val prácticamente nulo.

Este comportamiento muestra un **infraajuste total**, ya que el árbol no tiene capacidad de aprendizaje suficiente.

---

### 5. Conclusión del análisis de validación

El conjunto de validación indica que:

- Los modelos sin poda (`ccp_alpha = 0`) funcionan mejor.
- El valor de `min_samples_leaf` controla la complejidad de forma más suave que α.
- El mejor equilibrio entre tamaño del árbol y capacidad predictiva se consigue con:

### **min_samples_leaf = 20 y ccp_alpha = 0**

Esta combinación coincide con las tendencias teóricas del Tema 7 y
con los experimentos previos del Apartado 2, donde ya se había observado
que:

- los modelos con min_samples_leaf alto generalizan mejor,
- pero la poda global de CART es demasiado destructiva para este dataset.

La validación cruzada confirma que esta es la configuración óptima para avanzar al
entrenamiento final del modelo (train + validation) y su posterior evaluación en test.


In [16]:
# Entrenar con (train + validation)
import numpy as np

X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])

final_tree = DecisionTreeClassifier(
    min_samples_leaf=best_m,
    ccp_alpha=best_a,
    random_state=0
).fit(X_train_val, y_train_val)

preds_test = final_tree.predict(X_test)
f1_test = f1_score(y_test, preds_test, average="macro")

root_feat = final_tree.tree_.feature[0]
n_nodes = final_tree.tree_.node_count
n_leaves = final_tree.get_n_leaves()


# Convertir todo a tipos Python nativos
best_result = {
    "min_samples_leaf": int(best_m),
    "ccp_alpha": float(best_a),
    "root_feature": int(root_feat),
    "n_nodes": int(n_nodes),
    "n_leaves": int(n_leaves),
    "F1_test": float(f1_test)
}

df_best = pd.DataFrame([best_result])
df_best


,min_samples_leaf,ccp_alpha,root_feature,n_nodes,n_leaves,F1_test
0,20,0.0,10,1089,545,0.457741


# Conclusión del Apartado 3 — Selección de hiperparámetros con validación 60/20/20

En este apartado se ha dividido el dataset en entrenamiento (60%), validación (20%) y test (20%),
y se han evaluado 16 combinaciones de los hiperparámetros del árbol de decisión:

- `min_samples_leaf ∈ {1, 5, 10, 20}`
- `ccp_alpha ∈ {0.0, 0.001, 0.01, 0.05}`

---

## Mejor combinación según el conjunto de validación

La mejor combinación fue:

- **min_samples_leaf = 20**
- **ccp_alpha = 0.0**

con un **F1 de validación ≈ 0.4469**, el mayor de todas las configuraciones probadas.

Este resultado es coherente con los experimentos del apartado 2:

- Valores altos de `min_samples_leaf` reducen la profundidad del árbol y evitan el sobreajuste.
- Cualquier valor de `ccp_alpha > 0` provoca una poda excesiva del árbol y un fuerte infraajuste.

---

## Entrenamiento final y evaluación en test

Con los hiperparámetros óptimos se reentrenó el modelo usando **train + validation (80%)**
y se evaluó sobre el conjunto de **test (20%)**, obteniéndose:

- **F1_test = 0.4577**
- **Nodo raíz:** atributo 10  
- **Nodos totales:** 1089  
- **Hojas:** 545  

El árbol final mantiene una estructura suficientemente compleja para capturar el patrón
del dataset, pero evita el sobreajuste extremo observado cuando `min_samples_leaf = 1`.

---

## Conclusión global

El modelo que mejor generaliza en este problema es:

### **DecisionTreeClassifier(min_samples_leaf=20, ccp_alpha=0.0)**

Este equilibrio entre *profundidad controlada* (pre-poda) y *ausencia de poda global*
produce el mejor rendimiento tanto en validación como en test, cumpliendo exactamente
la metodología requerida de selección de hiperparámetros.
